# Building the Experiment 2 (similarity) stimuli

This notebook writes `materials/src/similarity-items.json`, the pair file
`similarity-experiment.js` loads at run time.

Experiment 2 asks participants to rate how similar two event descriptions are.
Every pair shares a verb. This holds the verb form fixed while sense and
specificity vary, allowing the experiment to measure within-verb aspectual
variation.

The similarity battery contains only contentful and bleached items, so the
notebook does not build templatic pairs. Calibration trials are defined in
`similarity-experiment.js` and prepended to each participant's timeline at run
time.

In [ ]:
import pandas as pd

## Load the sentence inventory

Unlike the proportion notebook, this one works from the sentence inventory
rather than from collected responses, because nothing about our pairing
depends on Experiment 1a. Each row is one PropBank sub-sense of one verb, carrying both its
contentful and its bleached surface form.

In [ ]:
data = pd.read_csv("../../data/manner-result-sentences.csv")

data

## Build the within-verb pairs

For each verb, we pair each new sub-sense with those already seen. This produces
each unordered pair once.

We emit each pair in up to three surface combinations: contentful with
contentful, contentful with bleached, and bleached with contentful. The latter two are checked against the pairs already generated for this verb,
because bleaching tends to collapse distinct senses onto the same sentence and
would otherwise yield a pair of identical strings.

**The same-sense flag.** A pair counts as same-sense when the two rows share
*both* a PropBank sense and a concreteness label. Recall that we split senses supporting both readings into concrete and
abstract sub-senses, so `hit.01-concrete` and `hit.01-abstract` count as
different senses here.
This flag is what we later use to separate within-sense from cross-sense
pairs.


In [ ]:
pairs = []

for (verb_type, verb), df_sense in data.groupby(["verb_type", "verb"]):
    sentences = []
    pairs_seen = []
    for _, r1 in df_sense.iterrows():
        for r2 in sentences:
            pair_same_sense = (
                r1.propbank_sense == r2.propbank_sense and 
                r1.concreteness == r2.concreteness
            )
            pair_metadata = [
                verb_type,
                verb,
                r1.propbank_sense,
                r1.concreteness,
                r1.propbank_sense + "-" + r1.concreteness,
                r2.propbank_sense,
                r2.concreteness,
                r2.propbank_sense + "-" + r2.concreteness,
            ]

            contentful_contentful_pair = [
                r1.sentence_contentful_episodic,
                r2.sentence_contentful_episodic
            ]

            pairs.append(
                pair_metadata + [
                    "contentful",
                    "contentful",
                    pair_same_sense
                ] + contentful_contentful_pair
            )

            pairs_seen.append(
                set(contentful_contentful_pair)
            )

            contentful_bleached_pair = [
                r1.sentence_contentful_episodic,
                r2.sentence_bleached_episodic
            ]

            if set(contentful_bleached_pair) not in pairs_seen:
                pairs.append(
                    pair_metadata + [
                        "contentful",
                        "bleached",
                        pair_same_sense
                    ] + contentful_bleached_pair
                )
                pairs_seen.append(set(contentful_bleached_pair))

            bleached_contentful_pair = [
                r1.sentence_bleached_episodic,
                r2.sentence_contentful_episodic
            ]

            if set(bleached_contentful_pair) not in pairs_seen:
                pairs.append(
                    pair_metadata + [
                        "contentful",
                        "bleached",
                        pair_same_sense
                    ] + bleached_contentful_pair
                )
                pairs_seen.append(
                    set(bleached_contentful_pair)
                )

        sentences.append(r1)


pair_col_names = [
    "verb_type",
    "verb",
    "propbank_sense1",
    "concreteness1",
    "sense1",
    "propbank_sense2",
    "concreteness2",
    "sense2"
]

pairs = pd.DataFrame(
    pairs,
    columns = pair_col_names + [
        "contentfulness1",
        "contentfulness2",
        "same_sense",
        "sentence1",
        "sentence2"
    ]
)


## Assign pairs to lists

We distribute pairs across 20 lists with different replication rates for the
two pair types.

We pair each list index `n1` with the index ten positions away, `n2`.
Contentful-contentful pairs go to *both*, so two lists' worth of participants
rate each of them, whereas contentful-bleached pairs go to only one of the
two, alternating by parity. The effect is to concentrate our replication on
the contentful-contentful pairs, which carry the within-sense comparisons that the
hypotheses concern, while still covering every contentful-bleached pair once.

The counters printed at the end report list length and the verb and pair-type
composition of each list.

In [ ]:
from collections import defaultdict, Counter

pair_items = [[] for _ in range(20)]


verb_counter = defaultdict(Counter)
contentfulness_counter = defaultdict(Counter)


list_num = 0

for verb, df_verb in pairs.groupby("verb"):
    for (cont1, cont2), df_cont in df_verb.groupby(["contentfulness1", "contentfulness2"]):
        for i, pair in df_cont.reset_index(drop=True).iterrows():
            n1 = list_num
            n2 = (list_num+int(len(pair_items)/2)) % len(pair_items)
            
            if cont1 == cont2: # include only half the bleached-contentful pairs
                pair_items[n1].append(
                    pair.to_dict()
                )

                verb_counter[n1][verb] += 1
                contentfulness_counter[n1][cont1, cont2] += 1

                pair_items[n2].append(
                    pair.to_dict()
                )

                verb_counter[n2][verb] += 1
                contentfulness_counter[n2][cont1, cont2] += 1
                
                list_num = (list_num + 1) % len(pair_items)

            elif list_num % 2:
                pair_items[n1].append(
                    pair.to_dict()
                )

                verb_counter[n1][verb] += 1
                contentfulness_counter[n1][cont1, cont2] += 1

                list_num = (list_num + 1) % len(pair_items)
            
            else:
                pair_items[n2].append(
                    pair.to_dict()
                )

                verb_counter[n2][verb] += 1
                contentfulness_counter[n2][cont1, cont2] += 1

                list_num = (list_num + 1) % len(pair_items)


print(f"Pairs lists:")

for i, l in enumerate(pair_items):
    print("List:      ", i)
    print("# of items:", len(l))
    print(verb_counter[i])
    print(contentfulness_counter[i])
    print()
    

## Verify the assignment

Here we check the three properties the previous cell is meant to guarantee:
every contentful-contentful pair appears exactly twice, every
contentful-bleached pair exactly once, and the total number of assignments
matches the number of pairs. The final expression re-derives the pair count by
weighting each contentful-contentful assignment as one half, providing an
independent check against double-counting.

These diagnostics print rather than assert. The expected output is
`(True, True, True)`, followed by two equal counts.

In [ ]:
item_counter = Counter()

for l in pair_items:
    for item in l:
        item_counter[(item["contentfulness1"], item["contentfulness2"], item["sentence1"], item["sentence2"])] += 1

(
    all(c == 2 for item, c in item_counter.items() if item[:2] == ("contentful", "contentful")), 
    all(c == 1 for item, c in item_counter.items() if item[:2] == ("contentful", "bleached")),
    sum(len(l) for l in pair_items) == pairs.shape[0]
)

pairs.shape[0], sum(
    0.5 if item["contentfulness1"] == item["contentfulness2"] else 1 
    for l in pair_items for item in l
)

## Write the stimulus file

This writes `materials/src/similarity-items.json`. As with our other item
files, we gitignore this one, so regenerating it will not show up as a repository
change.

In [ ]:
import json

with open("../src/similarity-items.json", "w") as f:
    json.dump(pair_items, f, indent=4)